### Shared functions — DST-aware window grid, ZSI, Harrell-Davis
Used via `%run ../../utils/zsi_functions`. Pure Python (no Spark) so the logic is easy to test.

In [0]:
import math
from datetime import date, datetime, timedelta, timezone
from datetime import time as dtime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from scipy.special import betainc   # regularized incomplete beta function I_x(a, b)


def hhmm_to_min(s):
    h, m = (int(x) for x in str(s).split(":"))
    return h * 60 + m


def wall_time_exists(t, tz):
    """False for a wall time inside the DST spring-forward gap (it does not survive wall -> UTC -> wall)."""
    back = t.replace(tzinfo=tz, fold=0).astimezone(timezone.utc).astimezone(tz).replace(tzinfo=None)
    return back == t


def window_exists(window_start, minutes, tz):
    """A window is in the grid if at least one real instant falls inside it.
    DST transitions happen on whole minutes, so checking every whole minute is exact."""
    return any(wall_time_exists(window_start + timedelta(minutes=j), tz) for j in range(minutes))


def build_grid(settings, start=None, end=None):
    """All 20-minute wall-clock windows of every operational day, with exists / morning flags."""
    tz = ZoneInfo(TIMEZONE)
    start = date.fromisoformat(start or OP_DATE_START)
    end = date.fromisoformat(end or OP_DATE_END)
    wm = int(settings["window_minutes"])
    op_start = hhmm_to_min(settings["op_day_start"])
    am_lo, am_hi = hhmm_to_min(settings["am_window_start_min"]), hhmm_to_min(settings["am_window_start_max"])
    assert 1440 % wm == 0, "window_minutes must divide 24h"

    rows, d = [], start
    while d <= end:
        day0 = datetime.combine(d, dtime()) + timedelta(minutes=op_start)
        for k in range(1440 // wm):
            ws = day0 + timedelta(minutes=k * wm)
            tod = (op_start + k * wm) % 1440
            exists = window_exists(ws, wm, tz)
            rows.append((d.isoformat(), k, ws.strftime("%Y-%m-%d %H:%M:%S"), f"{tod // 60:02d}:{tod % 60:02d}",
                         d.isoweekday(), exists,
                         exists and d.isoweekday() in settings["am_weekdays"] and am_lo <= tod <= am_hi))
        d += timedelta(days=1)
    return rows


GRID_COLUMNS = "operational_date STRING, window_idx INT, window_start STRING, window_start_tod STRING, " \
               "op_weekday INT, window_exists BOOLEAN, is_am BOOLEAN"


def zsi(P, D, p_offset=1, c=3):
    """ZSI(z,w) = sqrt(P) + sqrt(P + 1) - sqrt(4D + 3)"""
    return math.sqrt(P) + math.sqrt(P + p_offset) - math.sqrt(4 * D + c)


def hd_weights(n, p):
    """Harrell-Davis weights: W_i = I_{i/n}(a,b) - I_{(i-1)/n}(a,b), a=(n+1)p, b=(n+1)(1-p)"""
    a, b = (n + 1) * p, (n + 1) * (1 - p)
    return np.diff(betainc(a, b, np.arange(n + 1, dtype=float) / n))


def hd_quantile(values, p):
    x = np.sort(np.asarray(values, dtype=float))
    return float(np.dot(hd_weights(len(x), p), x))


def add_dq_columns(raw):
    """Helper columns (prefixed '_') used by the profile and the DQ rules. Never written to silver."""
    from pyspark.sql import Window
    from pyspark.sql import functions as F
    src_cols = [c for c in raw.columns if c != "processing_timestamp"]
    pu, do = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
    key = F.sha2(F.concat_ws("\u001f", *[F.coalesce(F.col(c).cast("string"), F.lit("␀NULL")) for c in src_cols]), 256)
    return (raw
            .withColumn("_pu_utc", F.to_utc_timestamp(pu, TIMEZONE))                    # real instant (DST-aware)
            .withColumn("_do_utc", F.to_utc_timestamp(do, TIMEZONE))
            .withColumn("_pu_nonexistent", F.from_utc_timestamp("_pu_utc", TIMEZONE) != pu)   # inside DST gap?
            .withColumn("_do_nonexistent", F.from_utc_timestamp("_do_utc", TIMEZONE) != do)
            .withColumn("_true_min", (F.col("_do_utc").cast("double") - F.col("_pu_utc").cast("double")) / 60)
            .withColumn("_naive_min", (do.cast("double") - pu.cast("double")) / 60)
            .withColumn("_trip_key", key)                                              # sha256 of all source columns
            .withColumn("_dup_rn", F.row_number().over(Window.partitionBy(*src_cols).orderBy("_trip_key"))))


def to_pandas_safe(df):
    """Collect a small Spark DataFrame; timestamps/dates become strings so no timezone conversion happens."""
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
    cols = [F.date_format(f.name, "yyyy-MM-dd HH:mm:ss").alias(f.name)
            if isinstance(f.dataType, (T.TimestampType, T.TimestampNTZType))
            else F.date_format(f.name, "yyyy-MM-dd").alias(f.name) if isinstance(f.dataType, T.DateType)
            else F.col(f.name) for f in df.schema.fields]
    return pd.DataFrame([r.asDict() for r in df.select(*cols).collect()], columns=df.columns)


def md_table(pdf):
    if len(pdf) == 0:
        return "_(no rows)_\n"
    fmt = lambda v: f"{v:.6g}" if isinstance(v, float) else ("NULL" if v is None else str(v))  # noqa: E731
    lines = ["| " + " | ".join(pdf.columns) + " |", "|" + "|".join("---" for _ in pdf.columns) + "|"]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in pdf.itertuples(index=False, name=None)]
    return "\n".join(lines) + "\n"


print("zsi_functions loaded")